# Analyse simplifiée des vues LLM

Notebook allégé pour diagnostiquer les vues générées par les LLM dans le pipeline Black-Litterman.

Contenu conservé :
- diagnostics mensuels des vues;
- similarité entre les modèles;
- graphique du `view_hit_rate`;
- graphique de la corrélation entre les vues et les rendements réalisés.

Aucune exportation CSV n'est effectuée dans ce notebook.


In [ ]:
from pathlib import Path
from itertools import combinations
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.options.display.float_format = "{:,.4f}".format

# ============================
# Configuration
# ============================

DATASET_PATH = Path("../data/filtered_sp25_data.csv")
RESPONSES_DIR = Path("../responses")

START = "2015-01-01"
END = "2025-06-30"

# Noms stricts attendus dans les fichiers de réponses :
# responses/{model}_{YYYY-MM-DD}_{YYYY-MM-DD}.json
MODELS = ["gpt", "gpt54mini", "gemma3", "llama", "qwen"]


## 1. Fonctions utilitaires

In [ ]:
def clean_ticker(x) -> str:
    if x is None:
        return ""
    s = str(x).strip().strip('"').strip("'").upper()
    if not s or s in {"NAN", "NONE"}:
        return ""
    if "-" in s and "." not in s:
        s = s.replace("-", ".")
    return "".join(s.split())


def parse_dates_yyyymmdd(s: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(s):
        ss = pd.to_numeric(s, errors="coerce").astype("Int64").astype(str).str.zfill(8)
        return pd.to_datetime(ss, format="%Y%m%d", errors="coerce")

    ss = s.astype(str).str.strip()
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    mask = ss.str.match(r"^\d{8}$", na=False)
    out.loc[mask] = pd.to_datetime(ss.loc[mask], format="%Y%m%d", errors="coerce")
    out.loc[~mask] = pd.to_datetime(ss.loc[~mask], errors="coerce")
    return out


def month_starts_between(start: str, end: str) -> pd.DatetimeIndex:
    return pd.date_range(pd.to_datetime(start), pd.to_datetime(end), freq="MS")


def month_end_from_period(p: pd.Period) -> pd.Timestamp:
    return p.to_timestamp(how="end").normalize()


def safe_corr(x, y) -> float:
    x = pd.Series(x, dtype=float)
    y = pd.Series(y, dtype=float)
    ok = x.notna() & y.notna() & np.isfinite(x) & np.isfinite(y)
    if ok.sum() < 3:
        return np.nan
    if x.loc[ok].std(ddof=1) <= 1e-15 or y.loc[ok].std(ddof=1) <= 1e-15:
        return np.nan
    return float(x.loc[ok].corr(y.loc[ok]))


def hit_rate(view, realized) -> float:
    view = pd.Series(view, dtype=float)
    realized = pd.Series(realized, dtype=float)
    ok = view.notna() & realized.notna() & np.isfinite(view) & np.isfinite(realized)
    ok &= (view != 0) & (realized != 0)
    if ok.sum() == 0:
        return np.nan
    return float((np.sign(view.loc[ok]) == np.sign(realized.loc[ok])).mean())


def market_regime(sp25_return: float) -> str:
    if pd.isna(sp25_return):
        return "unknown"
    if sp25_return >= 0.03:
        return "strong_up"
    if sp25_return > 0:
        return "up"
    if sp25_return <= -0.03:
        return "strong_down"
    return "down"


## 2. Chargement du dataset et benchmark S&P25

In [ ]:
def load_dataset(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    required = {"date", "tic", "stock_ret", "market_equity"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Colonnes manquantes dans {path}: {sorted(missing)}")

    df = df.copy()
    df["date_dt"] = parse_dates_yyyymmdd(df["date"])
    df["tic"] = df["tic"].map(clean_ticker)
    df["stock_ret"] = pd.to_numeric(df["stock_ret"], errors="coerce")
    df["market_equity"] = pd.to_numeric(df["market_equity"], errors="coerce")
    df = df.dropna(subset=["date_dt", "tic"]).copy()
    df = df[df["tic"] != ""].copy()
    df["ym"] = df["date_dt"].dt.to_period("M")
    df = df.sort_values(["ym", "tic", "date_dt"])
    return df.drop_duplicates(["ym", "tic"], keep="last")


def compute_sp25_returns(data: pd.DataFrame, start: str, end: str) -> tuple[pd.DataFrame, pd.Series]:
    returns_panel = data.pivot(index="ym", columns="tic", values="stock_ret").sort_index()
    mcap_panel = data.pivot(index="ym", columns="tic", values="market_equity").sort_index()

    weights = mcap_panel.where(mcap_panel > 0)
    weights = weights.div(weights.sum(axis=1), axis=0).fillna(0.0)

    out = {}
    for m0 in month_starts_between(start, end):
        weight_month = m0.to_period("M")
        target_month = (m0 + pd.offsets.MonthBegin(1)).to_period("M")

        if weight_month not in weights.index or target_month not in returns_panel.index:
            continue

        w = weights.loc[weight_month]
        r = returns_panel.loc[target_month]
        ok = w.notna() & r.notna() & np.isfinite(w) & np.isfinite(r)
        if ok.sum() == 0:
            continue

        denom = w.loc[ok].sum()
        if denom <= 1e-12:
            continue

        out[month_end_from_period(target_month)] = float((w.loc[ok] * r.loc[ok]).sum() / denom)

    return returns_panel, pd.Series(out, name="sp25_return").sort_index()


data = load_dataset(DATASET_PATH)
returns_panel, sp25_returns = compute_sp25_returns(data, START, END)

print("Dataset:", returns_panel.index.min(), "->", returns_panel.index.max(), "| tickers:", returns_panel.shape[1])
print("Benchmark S&P25:", sp25_returns.index.min(), "->", sp25_returns.index.max(), "| n:", len(sp25_returns))
display(sp25_returns.head())


## 3. Chargement des vues LLM

In [ ]:
def load_response_file(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    views = {}
    for ticker, values in raw.items():
        ticker = clean_ticker(ticker)
        if not ticker or not isinstance(values, dict):
            continue

        samples = values.get("expected_return")
        if not isinstance(samples, list):
            samples = [samples]

        s = pd.to_numeric(pd.Series(samples), errors="coerce").dropna()
        if s.empty:
            continue

        views[ticker] = {
            "view_mean": float(s.mean()),
            "view_median": float(s.median()),
            "view_std": float(s.std(ddof=1)) if len(s) > 1 else np.nan,
            "view_n": int(len(s)),
        }
    return views


def build_views_panel(models, responses_dir: Path, start: str, end: str) -> pd.DataFrame:
    rows = []
    missing = {m: 0 for m in models}
    loaded = {m: 0 for m in models}

    for m0 in month_starts_between(start, end):
        start_date = m0.strftime("%Y-%m-%d")
        end_date = (m0 + pd.offsets.MonthEnd(1)).strftime("%Y-%m-%d")
        target_month = (m0 + pd.offsets.MonthBegin(1)).to_period("M")
        target_date = month_end_from_period(target_month)

        for model in models:
            path = responses_dir / f"{model}_{start_date}_{end_date}.json"
            if not path.exists():
                missing[model] += 1
                continue

            loaded[model] += 1
            views = load_response_file(path)

            for ticker, vals in views.items():
                realized = np.nan
                if target_month in returns_panel.index and ticker in returns_panel.columns:
                    realized = returns_panel.loc[target_month, ticker]

                rows.append({
                    "model": model,
                    "view_date": m0,
                    "target_month": target_month,
                    "target_date": target_date,
                    "tic": ticker,
                    "realized_next_return": realized,
                    **vals,
                })

    if not rows:
        raise ValueError("Aucune vue chargée. Vérifie MODELS, START/END et RESPONSES_DIR.")

    for model in models:
        print(f"{model}: {loaded[model]} fichiers chargés | {missing[model]} mois manquants")

    views = pd.DataFrame(rows)
    views["view_sign"] = np.sign(views["view_mean"])
    views["realized_sign"] = np.sign(views["realized_next_return"])
    views["correct_sign"] = (
        views["view_sign"].ne(0)
        & views["realized_sign"].ne(0)
        & (views["view_sign"] == views["realized_sign"])
    )
    return views


views_long = build_views_panel(MODELS, RESPONSES_DIR, START, END)

print("Vues chargées:", views_long.shape)
display(views_long.head())


## 4. Diagnostic mensuel des vues

In [ ]:
monthly_rows = []

for (model, target_date), g in views_long.groupby(["model", "target_date"]):
    g = g.dropna(subset=["view_mean", "realized_next_return"]).copy()
    if g.empty:
        continue

    view = g.set_index("tic")["view_mean"]
    realized = g.set_index("tic")["realized_next_return"]
    sp25_return = sp25_returns.loc[target_date] if target_date in sp25_returns.index else np.nan

    monthly_rows.append({
        "model": model,
        "target_date": target_date,
        "n_views": len(g),
        "sp25_return": sp25_return,
        "market_regime": market_regime(sp25_return),
        "view_mean_avg": float(view.mean()),
        "view_median_avg": float(view.median()),
        "view_std_cross_section": float(view.std(ddof=1)) if len(view) > 1 else np.nan,
        "view_pos_frac": float((view > 0).mean()),
        "view_neg_frac": float((view < 0).mean()),
        "realized_avg": float(realized.mean()),
        "view_realized_corr": safe_corr(view, realized),
        "view_realized_rank_corr": safe_corr(view.rank(), realized.rank()),
        "view_hit_rate": hit_rate(view, realized),
    })

views_monthly = pd.DataFrame(monthly_rows).sort_values(["model", "target_date"]).reset_index(drop=True)

views_summary = (
    views_monthly.groupby("model")
    .agg(
        n_months=("target_date", "nunique"),
        avg_view_mean=("view_mean_avg", "mean"),
        avg_view_pos_frac=("view_pos_frac", "mean"),
        avg_view_std_cross_section=("view_std_cross_section", "mean"),
        avg_view_realized_corr=("view_realized_corr", "mean"),
        median_view_realized_corr=("view_realized_corr", "median"),
        avg_rank_corr=("view_realized_rank_corr", "mean"),
        avg_hit_rate=("view_hit_rate", "mean"),
    )
    .reset_index()
    .sort_values("avg_hit_rate", ascending=False)
)

display(views_monthly.head())
display(views_summary)


## 5. Graphiques : hit rate et corrélation vues-rendements

In [ ]:
def plot_monthly_metric(metric: str, title: str, baseline: float | None = None) -> None:
    plt.figure(figsize=(11, 5))

    for model in MODELS:
        d = views_monthly[views_monthly["model"] == model].sort_values("target_date")
        if d.empty:
            continue
        y = d[metric].rolling(6, min_periods=3).mean()
        plt.plot(d["target_date"], y, label=model)

    if baseline is not None:
        plt.axhline(baseline, linestyle="--", linewidth=1)

    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel(metric)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_monthly_metric(
    metric="view_hit_rate",
    title="Hit rate des vues - moyenne mobile 6 mois",
    baseline=0.5,
)

plot_monthly_metric(
    metric="view_realized_corr",
    title="Corrélation entre les vues et les rendements réalisés - moyenne mobile 6 mois",
    baseline=0.0,
)


## 6. Similarité entre les modèles

In [ ]:
inter_rows = []

for target_date, gdate in views_long.groupby("target_date"):
    models_present = sorted(gdate["model"].unique())

    for model_1, model_2 in combinations(models_present, 2):
        g1 = gdate[gdate["model"] == model_1].set_index("tic")
        g2 = gdate[gdate["model"] == model_2].set_index("tic")
        common = g1.index.intersection(g2.index)

        if len(common) < 3:
            continue

        v1 = g1.loc[common, "view_mean"]
        v2 = g2.loc[common, "view_mean"]

        inter_rows.append({
            "target_date": target_date,
            "model_1": model_1,
            "model_2": model_2,
            "pair": f"{model_1} vs {model_2}",
            "n_common": len(common),
            "view_corr": safe_corr(v1, v2),
            "view_rank_corr": safe_corr(v1.rank(), v2.rank()),
            "sign_agreement": float((np.sign(v1) == np.sign(v2)).mean()),
        })

inter_model_views = pd.DataFrame(inter_rows)

inter_summary = (
    inter_model_views.groupby("pair")
    .agg(
        n_months=("target_date", "nunique"),
        avg_n_common=("n_common", "mean"),
        avg_view_corr=("view_corr", "mean"),
        avg_rank_corr=("view_rank_corr", "mean"),
        avg_sign_agreement=("sign_agreement", "mean"),
    )
    .reset_index()
    .sort_values("avg_view_corr", ascending=False)
)

display(inter_summary)

corr_mat = pd.DataFrame(np.eye(len(MODELS)), index=MODELS, columns=MODELS, dtype=float)

for _, row in inter_summary.iterrows():
    model_1, _, model_2 = row["pair"].partition(" vs ")
    corr_mat.loc[model_1, model_2] = row["avg_view_corr"]
    corr_mat.loc[model_2, model_1] = row["avg_view_corr"]

plt.figure(figsize=(7, 5))
plt.imshow(corr_mat, aspect="auto")
plt.colorbar(label="Corrélation moyenne des vues")
plt.xticks(range(len(corr_mat.columns)), corr_mat.columns, rotation=45)
plt.yticks(range(len(corr_mat.index)), corr_mat.index)
plt.title("Similarité moyenne des vues entre modèles")

for i in range(len(corr_mat.index)):
    for j in range(len(corr_mat.columns)):
        val = corr_mat.iloc[i, j]
        if pd.notna(val):
            plt.text(j, i, f"{val:.2f}", ha="center", va="center")

plt.tight_layout()
plt.show()
